# 12b — View Cutouts for a Gaia-stable diaObject

## Purpose

Visualise the triplet of image stamps **(Science / Template / Difference / Science−Template)**
for every diaSource detection of a user-selected `diaObjectId`, in order to demonstrate
that **dipole residuals** in the PSF-subtracted difference image are responsible for
triggering spurious Rubin AP alerts on intrinsically stable Gaia stars.

## Layout — one row per diaSource

| Col | Image | Annotation |
|-----|-------|------------|
| 1 | **Science** (calexp) | scienceFlux [nJy], mag_AB |
| 2 | **Template** (coadd) | templateFlux [nJy], mag_AB |
| 3 | **Difference** (DIA, downloaded) | psfFlux [nJy]; mag_AB if psfFlux > 0 |
| 4 | **Science − Template** (computed here) | residual dipole pattern |

Row annotation: `visit | detector | band | date | SNR | isDipole`

## Data sources

| Data | Location |
|------|----------|
| Cutout `.npy` arrays + all metadata (incl. dipole flags) | `fullcutouts_{diaObjectId}/manifest.csv` |
| Forced photometry | `fullcutouts_{diaObjectId}/manifest_fp.csv` |
| Gaia DR3 object-level metadata | `data_FINK_BLOCK_LC_08b/fink_diaobj_gaia_join_matched.csv` |

---
- **Author:** Sylvie Dagoret-Campagne — IJCLab / IN2P3 / CNRS — Université Paris-Saclay
- **Creation date:** 2026-05-13
- **Subject:** Fink/LSST DIA — Dipole hypothesis — cutout visualisation


## 1. Imports & configuration

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.time import Time

warnings.filterwarnings("ignore")
print(f"pandas {pd.__version__}  |  numpy {np.__version__}")

In [ ]:
try:
    import ipympl

    %matplotlib widget
    print("ipympl → interactive backend")
except ImportError:
    %matplotlib inline
    print("inline backend")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# USER PARAMETERS  ← edit here
# ─────────────────────────────────────────────────────────────────────────────

# diaObjectId to inspect — must have a fullcutouts_{id}/ directory
# Available: 313761042226217038 | 313774269803790362 | 313888627193544815
# DIAOBJECT_ID = 313761042226217038
DIAOBJECT_ID = 313774269803790362

# Bands to display: None = all, or e.g. ["r", "i"]
BANDS_FILTER = None

# Max rows per band (None = unlimited)
MAX_ROWS_PER_BAND = None

# Image stretch: 'zscale' (percentile 1–99) or 'symlog'
STRETCH_MODE = "zscale"

# Color maps
CMAP_SCI = "gray"
CMAP_DIFF = "RdBu_r"  # diverging: red = positive, blue = negative

# Figure width (inches); row height is fixed at 3.2 in
FIG_WIDTH = 16

# Save figures to DIR_FIGS?
SAVE_FIGS = True
DIR_FIGS = "figs_FINK_BLOCK_LC_12b"

# ─────────────────────────────────────────────────────────────────────────────
# Fixed paths
# ─────────────────────────────────────────────────────────────────────────────
DIR_CUTOUTS = f"fullcutouts_{DIAOBJECT_ID}"
FILE_MANIFEST = os.path.join(DIR_CUTOUTS, "manifest.csv")
FILE_MANIFEST_FP = os.path.join(DIR_CUTOUTS, "manifest_fp.csv")
FILE_GAIA_MATCHED = os.path.join("data_FINK_BLOCK_LC_08b", "fink_diaobj_gaia_join_matched.csv")

AB_FLUX_ZERO = 3631e9  # nJy → m_AB = −2.5 × log10(f / AB_FLUX_ZERO)
BAND_ORDER = list("ugrizy")
BAND_COLORS = {"u": "#9b59b6", "g": "#2ecc71", "r": "#e74c3c", "i": "#e67e22", "z": "#3498db", "y": "#795548"}

os.makedirs(DIR_FIGS, exist_ok=True)
print(f"diaObjectId : {DIAOBJECT_ID}")
print(f"Manifest    : {os.path.abspath(FILE_MANIFEST)}")
print(f"Manifest fp : {os.path.abspath(FILE_MANIFEST_FP)}")
print(f"Figures     : {os.path.abspath(DIR_FIGS)}")

## 2. Utility functions

In [ ]:
def flux_to_mag_AB(flux_nJy):
    """Convert flux in nJy to AB magnitude. Returns NaN for non-positive flux."""
    try:
        f = float(flux_nJy)
    except (TypeError, ValueError):
        return np.nan
    return -2.5 * np.log10(f / AB_FLUX_ZERO) if (np.isfinite(f) and f > 0) else np.nan


def zscale(arr, lo=1.0, hi=99.0):
    """Return (vmin, vmax) from percentile-based z-scale."""
    finite = arr[np.isfinite(arr)]
    if len(finite) == 0:
        return 0.0, 1.0
    return float(np.percentile(finite, lo)), float(np.percentile(finite, hi))


def symvlim(arr):
    """Return symmetric ±vmax for a diverging image display."""
    finite = arr[np.isfinite(arr)]
    v = float(np.abs(finite).max()) if len(finite) else 1.0
    return -v, v


def load_npy(src_id, band, kind):
    """Load fullcutouts_{obj}/cutouts/{src_id}_{band}_{kind}.npy. Returns float32 or None."""
    fpath = os.path.join(DIR_CUTOUTS, "cutouts", f"{src_id}_{band}_{kind}.npy")
    return np.load(fpath).astype(np.float32) if os.path.exists(fpath) else None


def parse_bool(val):
    """Coerce a dipole flag (bool / int / str / NaN) to Python bool."""
    if isinstance(val, bool):
        return val
    if isinstance(val, (int, float)) and np.isfinite(float(val)):
        return bool(int(val))
    if isinstance(val, str):
        return val.strip().lower() in ("true", "1", "yes")
    return False


def mjd_to_date(mjd):
    """MJD (TAI) → ISO date string YYYY-MM-DD."""
    try:
        return Time(float(mjd), format="mjd", scale="tai").isot[:10]
    except Exception:
        return "?"


print("Utility functions defined.")

## 3. Load manifest (flux + dipole flags)

In [ ]:
if not os.path.exists(FILE_MANIFEST):
    raise FileNotFoundError(
        f"{FILE_MANIFEST} not found.\nRun: python fink_download_full_cutouts.py --obj_id {DIAOBJECT_ID}"
    )

df = pd.read_csv(FILE_MANIFEST)
df["isDipole"] = df["r:isDipole"].fillna(False).apply(parse_bool)
df = df.sort_values("r:midpointMjdTai").reset_index(drop=True)

n_dip = df["isDipole"].sum()
print(f"Manifest : {len(df)} diaSources  |  {n_dip} dipoles ({100 * n_dip / len(df):.1f}%)")
print(f"Bands    : {sorted(df['r:band'].unique())}")
df[
    [
        "r:diaSourceId",
        "r:band",
        "r:visit",
        "r:detector",
        "r:psfFlux",
        "r:scienceFlux",
        "r:templateFlux",
        "isDipole",
    ]
].head()

## 4. Load Gaia DR3 object-level metadata

In [ ]:
gaia_G_mag, gaia_status, fink_group, gaia_dr3_id = np.nan, "?", "?", "?"

if os.path.exists(FILE_GAIA_MATCHED):
    df_gaia = pd.read_csv(FILE_GAIA_MATCHED)
    hit = df_gaia[df_gaia["diaObjectId"].astype(str) == str(DIAOBJECT_ID)]
    if len(hit):
        r = hit.iloc[0]
        gaia_G_mag = float(r.get("gaia_phot_g_mean_mag", np.nan))
        gaia_status = str(r.get("gaia_status", "?"))
        fink_group = str(r.get("group", "?"))
        gaia_dr3_id = str(r.get("dr3_source_id", "?"))

G_str = f"{gaia_G_mag:.2f}" if np.isfinite(gaia_G_mag) else "?"
print(f"Gaia G={G_str}  |  status={gaia_status}  |  group={fink_group}  |  DR3={gaia_dr3_id}")

## 5. Cutout visualisation — one figure per band

In [ ]:
bands_available = [b for b in BAND_ORDER if b in df["r:band"].values]
bands_to_plot = [b for b in bands_available if BANDS_FILTER is None or b in BANDS_FILTER]
print(f"Bands to plot: {bands_to_plot}")


def show_image(ax, arr, cmap, stretch="zscale", diverging=False):
    """Display a 2D array on ax with chosen stretch and a colorbar."""
    if arr is None:
        ax.text(
            0.5, 0.5, "missing", ha="center", va="center", transform=ax.transAxes, color="red", fontsize=9
        )
        ax.axis("off")
        return
    vmin, vmax = symvlim(arr) if (diverging or stretch == "symlog") else zscale(arr)
    im = ax.imshow(arr, origin="lower", cmap=cmap, vmin=vmin, vmax=vmax)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cx = (ax.get_xlim()[0] + ax.get_xlim()[1]) / 2
    cy = (ax.get_ylim()[0] + ax.get_ylim()[1]) / 2
    ax.axvline(cx, color="yellow", lw=0.8, ls="--", alpha=0.7)
    ax.axhline(cy, color="yellow", lw=0.8, ls="--", alpha=0.7)
    ax.axis("off")


for band in bands_to_plot:
    df_band = df[df["r:band"] == band].sort_values("r:midpointMjdTai").reset_index(drop=True)
    if MAX_ROWS_PER_BAND:
        df_band = df_band.head(MAX_ROWS_PER_BAND)

    n_rows = len(df_band)
    n_dip_b = df_band["isDipole"].sum()
    bcolor = BAND_COLORS.get(band, "gray")
    print(f"Band {band}: {n_rows} rows  ({n_dip_b} dipoles)")

    fig, axes = plt.subplots(n_rows, 4, figsize=(FIG_WIDTH, 3.2 * n_rows), squeeze=False)
    fig.suptitle(
        f"diaObjectId={DIAOBJECT_ID}   band={band}   "
        f"{n_rows} detections  ({n_dip_b} dipoles)\n"
        f"Fink group: {fink_group}   Gaia G={G_str} mag   "
        f"status: {gaia_status}   DR3: {gaia_dr3_id}\n"
        "Science  |  Template  |  DIA Difference (downloaded)  |  Sci − Template (computed)",
        fontsize=10,
        y=1.005,
        color=bcolor,
        fontweight="bold",
    )

    for i, row in df_band.iterrows():
        src_id = str(row["r:diaSourceId"])
        visit = row.get("r:visit", "?")
        detector = row.get("r:detector", "?")
        mjd = float(row["r:midpointMjdTai"])
        snr = float(row.get("r:snr", np.nan))
        is_dip = bool(row["isDipole"])
        psf_f = float(row.get("r:psfFlux", np.nan))
        sci_f = float(row.get("r:scienceFlux", np.nan))
        tmp_f = float(row.get("r:templateFlux", np.nan))

        sci_arr = load_npy(src_id, band, "Science")
        tmp_arr = load_npy(src_id, band, "Template")
        dif_arr = load_npy(src_id, band, "Difference")
        res_arr = None
        if sci_arr is not None and tmp_arr is not None:
            ny = min(sci_arr.shape[0], tmp_arr.shape[0])
            nx = min(sci_arr.shape[1], tmp_arr.shape[1])
            res_arr = sci_arr[:ny, :nx] - tmp_arr[:ny, :nx]

        axs = axes[i]

        show_image(axs[0], sci_arr, CMAP_SCI, STRETCH_MODE)
        sci_m = flux_to_mag_AB(sci_f)
        axs[0].set_title(
            f"Science\n{sci_f:+.0f} nJy" + (f"  {sci_m:.2f} mag" if np.isfinite(sci_m) else ""), fontsize=8
        )

        show_image(axs[1], tmp_arr, CMAP_SCI, STRETCH_MODE)
        tmp_m = flux_to_mag_AB(tmp_f)
        axs[1].set_title(
            f"Template\n{tmp_f:+.0f} nJy" + (f"  {tmp_m:.2f} mag" if np.isfinite(tmp_m) else ""), fontsize=8
        )

        show_image(axs[2], dif_arr, CMAP_DIFF, STRETCH_MODE, diverging=True)
        psf_m = flux_to_mag_AB(psf_f)
        psf_mag_str = f"  {psf_m:.2f} mag" if (np.isfinite(psf_f) and psf_f > 0) else "  (neg flux)"
        axs[2].set_title(f"DIA Difference (downloaded)\n{psf_f:+.0f} nJy{psf_mag_str}", fontsize=8)

        show_image(axs[3], res_arr, CMAP_DIFF, STRETCH_MODE, diverging=True)
        axs[3].set_title("Sci − Template\n(computed here)", fontsize=8)

        dip_str = "\U0001f534 DIPOLE" if is_dip else "\u25e6 no-dipole"
        dip_color = "red" if is_dip else "black"
        axs[0].set_ylabel(
            f"#{i + 1}  visit {visit}\ndet {detector}  band {band}\n{mjd_to_date(mjd)}\nSNR={snr:.1f}",
            fontsize=7.5,
            rotation=0,
            labelpad=110,
            va="center",
        )
        axs[0].text(
            0.02,
            0.97,
            dip_str,
            transform=axs[0].transAxes,
            fontsize=8,
            fontweight="bold",
            color=dip_color,
            va="top",
            bbox=dict(boxstyle="round,pad=0.2", fc="white", ec=dip_color, alpha=0.75),
        )

    plt.tight_layout()
    if SAVE_FIGS:
        stem = f"cutouts_obj{DIAOBJECT_ID}_band{band}"
        for ext in ("pdf", "png"):
            plt.savefig(os.path.join(DIR_FIGS, f"{stem}.{ext}"), bbox_inches="tight", dpi=120)
        print(f"  → saved {stem}.{{pdf,png}}")
    plt.show()

## 6. Summary table

In [ ]:
from IPython.display import display as ipy_display

wanted = [
    "r:band",
    "r:visit",
    "r:detector",
    "r:midpointMjdTai",
    "r:snr",
    "r:psfFlux",
    "r:scienceFlux",
    "r:templateFlux",
    "isDipole",
    "r:dipoleFluxDiff",
    "r:dipoleLength",
    "r:dipoleAngle",
]
cols = [c for c in wanted if c in df.columns]

df_sum = df[cols].copy()
df_sum["mag_sci"] = df["r:scienceFlux"].apply(flux_to_mag_AB)
df_sum["mag_temp"] = df["r:templateFlux"].apply(flux_to_mag_AB)
df_sum["mag_psf"] = df["r:psfFlux"].apply(
    lambda f: flux_to_mag_AB(f) if (pd.notna(f) and float(f) > 0) else np.nan
)
df_sum["obs_date"] = df["r:midpointMjdTai"].apply(mjd_to_date)

print(f"diaObjectId={DIAOBJECT_ID}  |  Gaia G={G_str}  |  status={gaia_status}  |  group={fink_group}")
ipy_display(df_sum.sort_values(["r:band", "r:midpointMjdTai"]).reset_index(drop=True))

print("\nDipole fraction per band:")
for b in BAND_ORDER:
    db = df[df["r:band"] == b]
    if not len(db):
        continue
    nd = db["isDipole"].sum()
    print(f"  {b}  {nd:3d}/{len(db):3d}  ({100 * nd / len(db):5.1f}%)  {'\u2588' * int(nd / len(db) * 20)}")

## 7. psfFlux light curve — src (filled) + fp (open) + dipoles highlighted

- **Filled circles** `●` : real diaSource detections (`manifest.csv`)
- **Open circles** `○` : forced photometry (`manifest_fp.csv`)
- **Filled red diamonds** `◆` : diaSource detections flagged `isDipole=True`

In [ ]:
# ── Load forced photometry ────────────────────────────────────────────────────
if os.path.exists(FILE_MANIFEST_FP):
    df_fp = pd.read_csv(FILE_MANIFEST_FP).sort_values("r:midpointMjdTai").reset_index(drop=True)
    print(f"Forced photometry : {len(df_fp)} points  bands: {sorted(df_fp['r:band'].unique())}")
else:
    df_fp = pd.DataFrame()
    print(f"No manifest_fp.csv found in {DIR_CUTOUTS}/ — fp not shown.")

# ── Common time origin (earliest point across src + fp) ──────────────────────
t0_candidates = [df["r:midpointMjdTai"].min()]
if not df_fp.empty:
    t0_candidates.append(df_fp["r:midpointMjdTai"].min())
t0 = min(t0_candidates)

bands_lc = [b for b in BAND_ORDER if b in df["r:band"].values]

fig, axes = plt.subplots(
    len(bands_lc),
    1,
    figsize=(12, 2.8 * len(bands_lc)),
    sharex=True,
    squeeze=False,
)

for ax_i, band in enumerate(bands_lc):
    ax = axes[ax_i][0]
    bc = BAND_COLORS.get(band, "gray")

    # ── diaSource detections ──────────────────────────────────────────────
    db = df[df["r:band"] == band].sort_values("r:midpointMjdTai")
    dt = db["r:midpointMjdTai"].values.astype(float) - t0
    psf = db["r:psfFlux"].values.astype(float)
    err = db["r:psfFluxErr"].values.astype(float)
    dip = db["isDipole"].values.astype(bool)

    # Non-dipole src — filled circles
    ax.errorbar(
        dt[~dip],
        psf[~dip],
        yerr=err[~dip],
        fmt="o",
        ms=5,
        color=bc,
        ecolor=bc,
        elinewidth=0.7,
        capsize=2,
        alpha=0.85,
        label=f"{band} src",
    )
    # Dipole src — filled red diamonds
    if dip.any():
        ax.errorbar(
            dt[dip],
            psf[dip],
            yerr=err[dip],
            fmt="D",
            ms=7,
            color="red",
            ecolor="red",
            elinewidth=1.2,
            capsize=3,
            alpha=0.95,
            label=f"{band} src DIPOLE",
        )

    # ── Forced photometry — open circles ─────────────────────────────────
    if not df_fp.empty and band in df_fp["r:band"].values:
        dbf = df_fp[df_fp["r:band"] == band].sort_values("r:midpointMjdTai")
        dtf = dbf["r:midpointMjdTai"].values.astype(float) - t0
        psfF = dbf["r:psfFlux"].values.astype(float)
        errF = dbf["r:psfFluxErr"].values.astype(float)
        ax.errorbar(
            dtf,
            psfF,
            yerr=errF,
            fmt="o",
            ms=5,
            markerfacecolor="none",  # open marker
            markeredgecolor=bc,
            markeredgewidth=1.2,
            ecolor=bc,
            elinewidth=0.7,
            capsize=2,
            alpha=0.7,
            label=f"{band} fp",
        )

    ax.axhline(0, color="k", lw=0.6, ls=":", alpha=0.4)
    ax.set_ylabel(f"psfFlux [nJy]\nband {band}", fontsize=9)
    ax.legend(fontsize=8, ncol=3)
    ax.grid(True, alpha=0.3)

axes[-1][0].set_xlabel(f"\u0394t [days from MJD {t0:.2f}]", fontsize=9)
fig.suptitle(
    f"psfFlux — diaObjectId={DIAOBJECT_ID}  Gaia G={G_str}  "
    f"status={gaia_status}  group={fink_group}\n"
    "\u25cf src (filled)   \u25cb fp (open)   \u25c6 src DIPOLE (red filled)",
    fontsize=10,
    y=1.01,
)
plt.tight_layout()

if SAVE_FIGS:
    stem = f"lc_dipoles_obj{DIAOBJECT_ID}"
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{stem}.{ext}"), bbox_inches="tight", dpi=120)
    print(f"\u2192 saved {stem}.{{pdf,png}}")

plt.show()